## 12-rag-revision

In [1]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [2]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [3]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [4]:
assistant.rag("How do I run Ollama locally?")

'To run Ollama locally:\n\n1. Install Ollama from: https://ollama.com/download  \n   - macOS: download the `.pkg`\n   - Windows: download the `.msi`\n   - Linux: run:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. Open a terminal and run:\n   ```bash\n   ollama run llama3\n   ```\n\nThis will download the LLaMA 3 model, start it locally, and open a chat-like interface.\n\nIf you want to test that the local server is running, use:\n```bash\ncurl http://localhost:11434\n```\n\nYou should get a response like:\n```json\n{"models": [...]}  \n```'

In [5]:
assistant.rag("How do I run Olama locally?")

'I don’t see any FAQ entry about running **Ollama** locally.\n\nThe closest related FAQ says you can run the course locally instead of Codespaces if you’re comfortable setting up Python, `uv`, Jupyter, Docker, and any other tools needed for the module, and that you should document your setup to keep it reproducible.\n\nIf you want, I can help you interpret what local setup would be needed based on that guidance.'

The word "Olama" doesn't match "Ollama" in our index. We use lexical search, so it looks for the exact word and finds nothing. The LLM gets these bad results and either says "I don't know" or answers with irrelevant information.

This is the limitation of a fixed pipeline. The search runs once with the exact query the user typed, and there's no second chance. The pipeline doesn't know the search failed, so it can't try again with a corrected query.

We need something smarter. We need an agent.